在数据预处理中，**异常值（Outliers）**检测是保证模型稳健性的关键步骤。如果数据中混入了因实验误差、读数错误产生的异常点，会严重偏差回归曲线或聚类中心。

以下是数学建模中最常用的两种统计学检测方法：**$3\sigma$ 原则**（基于参数统计）和 **箱线图（Boxplot）**（基于非参数统计）。

---

### 一、 正态分布 $3\sigma$ 原则 (Z-Score 方法)

#### 1. 算法原理与数学推导
**核心思想**：如果数据服从正态分布，那么绝大多数数值应该集中在均值附近，超出一定偏差范围的点被视为极小概率事件。

**数学推导**：
设随机变量 $X \sim N(\mu, \sigma^2)$，其概率密度函数为典型的钟形曲线。根据正态分布的累积概率计算：
*   $P(\mu - \sigma < X < \mu + \sigma) \approx 68.27\%$
*   $P(\mu - 2\sigma < X < \mu + 2\sigma) \approx 95.45\%$
*   **$P(\mu - 3\sigma < X < \mu + 3\sigma) \approx 99.73\%$**

**逻辑推论**：
在一次随机试验中，数值落在 $(\mu-3\sigma, \mu+3\sigma)$ 之外的概率仅为 **0.27%**。在数学建模的实际操作中，这被视为“小概率事件”，根据统计学原理，小概率事件在单次试验中几乎不会发生，若发生则判定为异常值。

#### 2. 适用场景
*   **前提条件**：数据必须近似服从**正态分布**。
*   **样本量**：通常要求样本量 $n > 30$。
*   **典型应用**：工业生产中的尺寸误差、人的身高/体重分布、标准化考试成绩。

#### 3. Python 代码框架
```python
import numpy as np
import pandas as pd

def detect_outliers_3sigma(data):
    """
    3sigma原则检测异常值
    :param data: 一维数组或Series
    :return: 异常值的索引, 异常值本身
    """
    mean_val = np.mean(data)
    std_val = np.std(data)

    # 计算上下界
    lower_bound = mean_val - 3 * std_val
    upper_bound = mean_val + 3 * std_val

    # 筛选异常值
    outliers_mask = (data < lower_bound) | (data > upper_bound)
    return np.where(outliers_mask)[0], data[outliers_mask]

# --- 示例 ---
data = pd.Series([10, 12, 11, 13, 12, 11, 100, 11, 12, 11]) # 100 是明显的异常
idx, val = detect_outliers_3sigma(data)
print(f"3sigma检测到的异常值索引: {idx}, 数值: {val.values}")
```

---

### 二、 箱线图检测法 (Tukey's Fences)

#### 1. 算法原理与数学推导
**核心思想**：利用数据的**分位数（Quartiles）**来定义边界，不需要假设数据服从任何特定的分布。

**数学推导**：
1.  **Q1 (下四分位数)**：数据从小到大排列后，第 25% 位置的数值。
2.  **Q3 (上四分位数)**：数据从小到大排列后，第 75% 位置的数值。
3.  **IQR (四分位距)**：$IQR = Q3 - Q1$，反映了中间 50% 数据的波动范围。
4.  **构建内栏（Inner Fences）**：
    *   **下限 (Lower Limit)**：$Q1 - 1.5 \times IQR$
    *   **上限 (Upper Limit)**：$Q3 + 1.5 \times IQR$

**为什么是 1.5 倍？**
在正态分布下，$\pm 1.5 \times IQR$ 约等于 $\pm 2.7\sigma$。这是一个经验值，既能捕捉到明显的离群点，又不会对略微偏离中心的数据过于敏感。

#### 2. 适用场景
*   **非正态数据**：数据呈现偏态（Skewed）或长尾分布时，箱线图比 $3\sigma$ 稳健得多。
*   **小样本**：对样本量要求不高。
*   **典型应用**：房价分布、公司薪资水平（通常是偏态增长）、金融收益率。

#### 3. Python 代码框架
```python
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def detect_outliers_boxplot(data):
    """
    箱线图方法检测异常值
    """
    Q1 = np.percentile(data, 25)
    Q3 = np.percentile(data, 75)
    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers_mask = (data < lower_limit) | (data > upper_limit)
    return np.where(outliers_mask)[0], data[outliers_mask]

# --- 可视化展示 ---
data = np.random.normal(10, 2, 100)
data = np.append(data, [25, -5]) # 手动添加异常点

plt.figure(figsize=(6, 4))
sns.boxplot(data=data)
plt.title("Boxplot Outlier Detection")
plt.show()

idx, val = detect_outliers_boxplot(data)
print(f"箱线图检测到的异常值: {val}")
```

---

### 三、 论文写作与“修修补补”建议

在数学建模论文中，异常值处理不能直接删掉，需要有一套完整的**说辞**：

#### 1. 如何选择方法？
*   如果你的论文第一步做了**正态性检验**（如 K-S 检验或 Shapiro-Wilk 检验）且通过了，请务必使用 **$3\sigma$ 原则**，这能体现出你对数据分布特征的深入利用。
*   如果数据分布不明或明显偏斜（比如收入数据），请直接写“**考虑到数据呈现非对称性分布，为提高鲁棒性，本文采用箱线图准则（IQR法）进行离群点检测。**”

#### 2. 检测到异常值后怎么办？
千万不要只写“删除”。在建模中，有三种处理策略：
1.  **删除法**：如果确定是记录错误或传感器故障。
2.  **盖帽法（Winsorization）**：将超出上限的数改为上限值，低于下限的数改为下限值（适合金融数据，保留样本量）。
3.  **插值法**：将异常值视为缺失值，利用你刚刚学过的**拉格朗日插值**或**均值**来替换。

#### 3. 论文表述示例：
> “初步分析发现原始序列存在明显的波动离群点。为消除异常数据对后续回归分析的负面影响，本文绘制了数据的箱线图（见图1）。根据四分位距（IQR）原则，我们将落在 $[Q1-1.5IQR, Q3+1.5IQR]$ 范围之外的 5 个数据点判定为异常值，并采用均值修复法对其进行了平滑处理。”

**下一类数据预处理算法（如：PCA 主成分分析、标准化/归一化、数据离散化），你想听哪一个的数学推导？**